# Министерство науки и высшего образования Российской Федерации
## Федеральное государственное бюджетное образовательное учреждение высшего образования
## Уфимский университет науки и технологии
### Кафедра АСУ

# Отчёт по практической работе №5
## Разведочный анализ данных (EDA) на Python
### по дисциплине «Интеллектуальные информационные системы»

**Выполнил:** ст. гр. ЭАС-401С Вахапов А.А.  
**Проверил:** Насыров Р.В.  

Уфа, 2026


# Цель работы

Освоить методы первичного исследования и визуализации данных на Python для подготовки данных к построению аналитических моделей.

# Исходные данные

Проанализирован набор **California Housing** из `scikit-learn`: 20 640 районов Калифорнии по данным переписи 1990 года. Использованы восемь признаков и целевая переменная `MedHouseVal`. Стоимость измеряется в сотнях тысяч долларов, доход `MedInc` — в десятках тысяч долларов. Набор загружен из официального архива California Housing; пропуски проверены фактическим подсчётом.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats.mstats import winsorize
from sklearn.datasets import fetch_california_housing

sns.set_theme(style="whitegrid")
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(f"Размер датасета: {df.shape}")
print(df.isna().sum())


Размер датасета: (20640, 9)
Пропусков по всем 9 переменным: 0


# Первичный осмотр и описательная статистика

Размерность набора — **20 640 × 9**. Все переменные числовые. Пропусков не обнаружено, поэтому заполнение или удаление строк не требовалось. Статистики ниже рассчитаны по полному набору наблюдений.


In [ ]:
df.info()
display(df.describe().T)
print("Асимметрия:")
print(df.skew())


| Переменная | Среднее | Медиана | Ст. откл. | Мин. | Q1 | Q3 | Макс. | Асимметрия |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Longitude | -119.570 | -118.490 | 2.004 | -124.350 | -121.800 | -118.010 | -114.310 | -0.298 |
| Latitude | 35.632 | 34.260 | 2.136 | 32.540 | 33.930 | 37.710 | 41.950 | 0.466 |
| HouseAge | 28.639 | 29.000 | 12.586 | 1.000 | 18.000 | 37.000 | 52.000 | 0.060 |
| AveRooms | 5.429 | 5.229 | 2.474 | 0.846 | 4.441 | 6.052 | 141.909 | 20.696 |
| AveBedrms | 1.097 | 1.049 | 0.474 | 0.333 | 1.006 | 1.100 | 34.067 | 31.315 |
| Population | 1425.477 | 1166.000 | 1132.462 | 3.000 | 787.000 | 1725.000 | 35682.000 | 4.935 |
| AveOccup | 3.071 | 2.818 | 10.386 | 0.692 | 2.430 | 3.282 | 1243.333 | 97.632 |
| MedInc | 3.871 | 3.535 | 1.900 | 0.500 | 2.563 | 4.743 | 15.000 | 1.647 |
| MedHouseVal | 2.069 | 1.797 | 1.154 | 0.150 | 1.196 | 2.647 | 5.000 | 0.978 |


# Визуализация распределений

Для всех переменных построены гистограммы и ящики с усами. Подписи осей содержат названия показателей и единицы измерения. Для `MedHouseVal` дополнительно построен Q–Q график.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, column in zip(axes.flat, df.columns):
    sns.histplot(df[column], bins=50, ax=ax, color="#1d4ed8")
    ax.set_title(column)
plt.tight_layout()

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, column in zip(axes.flat, df.columns):
    sns.boxplot(y=df[column], ax=ax, color="#93c5fd")
    ax.set_title(column)
plt.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df["MedHouseVal"], kde=True, ax=axes[0], color="#1d4ed8")
stats.probplot(df["MedHouseVal"], dist="norm", plot=axes[1])
plt.tight_layout()


![Гистограммы распределений](figures/01_distributions.svg)

*Рисунок 1. Распределения переменных California Housing.*

![Ящики с усами](figures/02_boxplots.svg)

*Рисунок 2. Ящики с усами. Для читаемости отдельно указано число выбросов по правилу IQR.*

![Q-Q график](figures/03_qq_medhouseval.svg)

*Рисунок 3. Q–Q график для целевой переменной.*


Распределения большинства счётных и средних признаков имеют правую асимметрию. Особенно выделяются `AveRooms` и `AveBedrms`: редкие районы с очень большими средними значениями создают длинный правый хвост. Q–Q график `MedHouseVal` отклоняется от теоретической прямой, а верхняя граница целевой переменной формирует заметное накопление значений. Следовательно, допущение нормальности для исходной целевой переменной не подтверждается визуально.

# Корреляционный анализ

Матрица построена по коэффициенту корреляции Пирсона. Наиболее сильная линейная связь с ценой получена у `MedInc` (`r = 0.688`).


In [ ]:
corr_matrix = df.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Матрица корреляций")

sns.pairplot(df[["MedInc", "AveRooms", "HouseAge", "MedHouseVal"]], corner=True)
plt.figure(figsize=(8, 6))
sns.regplot(x="MedInc", y="MedHouseVal", data=df, scatter_kws={"alpha": 0.3}, line_kws={"color": "#c2410c"})
plt.title("Зависимость цены от медианного дохода")


![Матрица корреляций](figures/04_correlation_heatmap.svg)

*Рисунок 4. Матрица корреляций Пирсона.*

![Парные графики](figures/05_pairplot.svg)

*Рисунок 5. Парные графики ключевых признаков.*

![Связь дохода и цены](figures/06_medinc_price.svg)

*Рисунок 6. Зависимость цены жилья от медианного дохода. Оранжевая линия — линейная регрессия.*


Помимо `MedInc`, связи с ценой заметно слабее. Для координат получены `r = -0.144` по широте и `r = -0.046` по долготе, что подтверждает наличие пространственного фактора, но не описывает его полностью линейной зависимостью. Между `AveRooms` и `AveBedrms` наблюдается высокая корреляция `r = 0.848`; при построении линейной модели этот факт следует учитывать как возможную мультиколлинеарность.

# Выбросы и преобразование данных


In [ ]:
def detect_outliers_iqr(data, column):
    q1, q3 = data[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return data[(data[column] < lower_bound) | (data[column] > upper_bound)], lower_bound, upper_bound

outliers, lower_bound, upper_bound = detect_outliers_iqr(df, "AveRooms")
z_scores = np.abs(stats.zscore(df["AveRooms"]))
outliers_z = df[z_scores > 3]
print(f"IQR: {len(outliers)} выбросов; границы [{lower_bound:.3f}, {upper_bound:.3f}]")
print(f"Z-метод (|z| > 3): {len(outliers_z)} выбросов")

df["AveRooms_winsorized"] = winsorize(df["AveRooms"], limits=[0.01, 0.01])
df["AveRooms_log"] = np.log1p(df["AveRooms"])
print(df[["AveRooms", "AveRooms_winsorized", "AveRooms_log"]].describe().T)


IQR: 511 выбросов; границы [2.023, 8.470]
Z-метод (|z| > 3): 133 выбросов


![Обработка AveRooms](figures/07_ave_rooms_treatment.svg)

*Рисунок 7. Исходное распределение, винсоризация 1% с каждого края и логарифмическое преобразование `AveRooms`.*


По IQR для `AveRooms` обнаружено **511** выбросов, по Z-методу при пороге `|z| > 3` — **133**. Автоматически удалять их не следует: такие районы могут отражать редкие, но реальные особенности жилого фонда. Винсоризация показана как способ снизить влияние экстремумов, а `log1p` — как преобразование, уменьшающее правую асимметрию. Для основной таблицы исходные значения не заменялись.

# Выводы

1. Набор содержит 20 640 наблюдений и 9 числовых переменных. Пропущенные значения отсутствуют.
2. Распределения ряда признаков несимметричны. Наиболее выраженный правый хвост наблюдается у средних характеристик помещений и населения.
3. В `AveRooms` выявлены статистические выбросы. Их следует проверять в предметном контексте, а не удалять безусловно.
4. Медианный доход — самый сильный линейный фактор цены жилья. Координаты также связаны с ценой, поэтому пространственные признаки полезны для дальнейшей модели.
5. Высокая связь `AveRooms` и `AveBedrms` требует контроля мультиколлинеарности при регрессионном моделировании.
6. Для последующего моделирования следует рассмотреть преобразование скошенных признаков, устойчивые к выбросам методы и нелинейное использование географических координат.

# Приложение: воспроизводимость

Ноутбук использует `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy` и `scikit-learn`. Для повторного запуска необходимо установить зависимости из `requirements.txt`; затем выполнить все ячейки сверху вниз.
